In [ ]:
%pip install pykokoro soundfile markdown beautifulsoup4 google-api-python-client

In [ ]:
import re
import subprocess
from pathlib import Path
import soundfile as sf
import markdown
from bs4 import BeautifulSoup
import numpy as np
import os
from pykokoro import KokoroPipeline, PipelineConfig
from pykokoro.generation_config import GenerationConfig

# Maps Unicode subscript digits and common subscript letters to ASCII equivalents.
# Pandoc preserves these from DOCX math notation (e.g. x₁, wₙ).
_SUBSCRIPT_MAP = str.maketrans("₀₁₂₃₄₅₆₇₈₉ₙₗ", "0123456789nl")

# Math symbols that have natural spoken equivalents.
_MATH_MAP = {"∑": "sum of", "⋯": "...", "·": " times ", "−": "-", "×": "x"}

def docx_to_markdown(docx_path):
    md_path = str(Path(docx_path).with_suffix('.md'))
    try:
        subprocess.run(
            ["pandoc", "-s", docx_path, "-t", "markdown", "-o", md_path],
            check=True,
            capture_output=True,
            text=True,
        )
    except FileNotFoundError:
        raise RuntimeError(
            "pandoc is not installed or not on PATH. "
            "Install it from https://pandoc.org/installing.html and restart the kernel."
        )
    print(f"Pandoc converted '{docx_path}' -> '{md_path}'")
    return md_path

def resolve_input(file_path):
    suffix = Path(file_path).suffix.lower()
    if suffix == '.docx':
        return docx_to_markdown(file_path)
    elif suffix == '.md':
        return file_path
    else:
        raise ValueError(f"Unsupported file type '{suffix}'. Pass a .docx or .md file.")

def preprocess_for_tts(text):
    """Cleans pandoc-generated markdown so it reads naturally as spoken audio.

    Pandoc grid tables, image references, emoji, and Unicode math notation all
    produce either silence or awkward literal readings in TTS. This strips or
    normalises each artifact before the text is fed to the markdown parser.
    """
    # Remove image reference lines (with or without pandoc attribute block)
    text = re.sub(r'!\[.*?\]\(.*?\)\{.*?\}\n?', '', text)
    text = re.sub(r'!\[.*?\]\(.*?\)\n?', '', text)

    # Strip pandoc grid table border lines: +----+ and +====+
    text = re.sub(r'^\+[-=]+\+.*\n?', '', text, flags=re.MULTILINE)

    # Strip leading pipe from callout box content lines, keeping the text
    text = re.sub(r'^\|\s*', '', text, flags=re.MULTILINE)

    # Drop all non-ASCII characters — removes emoji, special arrows, etc.
    text = text.encode('ascii', 'ignore').decode('ascii')

    # Normalise Unicode subscript digits/letters to plain ASCII
    text = text.translate(_SUBSCRIPT_MAP)

    # Replace math symbols with speakable words
    for sym, word in _MATH_MAP.items():
        text = text.replace(sym, word)

    return text

def parse_markdown(file_path):
    """Converts a Markdown file into clean, plain text suitable for TTS."""
    with open(file_path, 'r', encoding='utf-8') as f:
        raw = f.read()

    cleaned = preprocess_for_tts(raw)
    html = markdown.markdown(cleaned)
    return ''.join(BeautifulSoup(html, "html.parser").findAll(string=True))

def generate_audio(file_path):
    md_path = resolve_input(file_path)
    text = parse_markdown(md_path)

    # Chunk the text by paragraphs.
    chunks = [chunk.strip() for chunk in text.split('\n\n') if chunk.strip()]

    # Initialize Kokoro with CPU
    config = PipelineConfig(
        voice="af_bella",
        provider="cpu",
        model_quality="fp32",
        generation=GenerationConfig(lang="en-us")
    )
    pipe = KokoroPipeline(config)

    print(f"Starting CPU audio generation for {len(chunks)} chunks...")
    audio_arrays = []
    sample_rate = 24000  # Kokoro's native output rate

    for i, chunk in enumerate(chunks):
        print(f"Processing chunk {i+1}/{len(chunks)}...")
        res = pipe.run(chunk)
        if res.audio is not None:
            audio_arrays.append(res.audio)
            sample_rate = res.sample_rate

    # Stitch chunks and save locally
    final_audio = np.concatenate(audio_arrays)
    output_filename = Path(file_path).stem + ".mp3"
    sf.write(output_filename, final_audio, sample_rate)
    print(f"Saved local file to: {output_filename}")

    return output_filename


if __name__ == "__main__":
    # Change to your .docx or .md file path — pandoc runs automatically for .docx
    input_file = "02_Training_Deep_Networks.docx"
    if os.path.exists(input_file):
        generated_audio = generate_audio(input_file)
    else:
        print(f"Error: {input_file} not found.")